# CS3807 Deep Learning Laboratory - Part 3: Model Construction and Training
## Experiments 1 & 2: Attention Type and Position Ablations

In [ ]:
# Cell 1: Load all prerequisites
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle

# Load data
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']
class_names = splits['class_names']
num_classes = splits['num_classes']
train_augment = splits['train_augment']

print(f"✓ Data loaded")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# Cell 2: Import attention modules (from Part 2)
# Here we redefine them for standalone execution
# (In practice, use from Part 2)

# Channel Attention
class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        
    def build(self, input_shape):
        channels = input_shape[-1]
        reduced = max(channels // self.reduction_ratio, 1)
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(reduced, activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        super().build(input_shape)
        
    def call(self, x):
        z = self.gap(x)
        z = self.fc1(z)
        z = self.fc2(z)
        return x * z

# Spatial Attention  
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.conv = layers.Conv2D(1, kernel_size, padding='same', activation='sigmoid')
        
    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg, mx], axis=-1)
        att = self.conv(concat)
        return x * att

# Self Attention
class SelfAttention(layers.Layer):
    def __init__(self, dim_scale=8, **kwargs):
        super().__init__(**kwargs)
        self.dim_scale = dim_scale
        
    def build(self, shape):
        self.scaled_dim = max(shape[-1] // self.dim_scale, 1)
        self.q = layers.Dense(self.scaled_dim)
        self.k = layers.Dense(self.scaled_dim)
        self.v = layers.Dense(self.scaled_dim)
        self.out_proj = layers.Dense(shape[-1])
        self.gamma = tf.Variable(tf.zeros((1,)), trainable=True)
        super().build(shape)
        
    def call(self, x):
        b, h, w, c = tf.shape(x)[0], x.shape[1], x.shape[2], x.shape[3]
        x_flat = tf.reshape(x, [b, h*w, c])
        q, k, v = self.q(x_flat), self.k(x_flat), self.v(x_flat)
        att = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(tf.cast(self.scaled_dim, tf.float32)))
        out = tf.matmul(att, v)
        out = self.out_proj(out)
        out = tf.reshape(out, [b, h, w, c])
        return x + self.gamma * out

print("✓ Attention modules loaded")

In [ ]:
# Cell 2b: Extended attention modules (SE / ECA / CBAM / Coordinate / Variable-Spatial)
# Standalone definitions so Part 3 runs without importing Part 2
class SEAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
    def build(self, input_shape):
        c = int(input_shape[-1]); r = max(c // self.reduction_ratio, 1)
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(r, activation='relu')
        self.fc2 = layers.Dense(c, activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        return x * self.fc2(self.fc1(self.gap(x)))
class ECAAttention(layers.Layer):
    def __init__(self, gamma=2, b=1, **kwargs):
        super().__init__(**kwargs)
        self.gamma, self.b = gamma, b
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
    def build(self, input_shape):
        import math
        c = int(input_shape[-1])
        k = int(abs(math.log2(c) / self.gamma + self.b / self.gamma))
        k = k if k % 2 == 1 else k + 1
        self.conv1d = layers.Conv1D(1, kernel_size=k, padding='same',
                                  use_bias=False, activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        z = tf.squeeze(self.gap(x), axis=1)
        return x * tf.expand_dims(self.conv1d(z), axis=1)
class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio, self.spatial_kernel = reduction_ratio, spatial_kernel
    def build(self, input_shape):
        c = int(input_shape[-1]); r = max(c // self.reduction_ratio, 1)
        self.fc1 = layers.Dense(r, activation='relu'); self.fc2 = layers.Dense(c)
        self.sp = layers.Conv2D(1, self.spatial_kernel, padding='same', activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        avg = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        mx = tf.reduce_max(x, axis=[1, 2], keepdims=True)
        cw = tf.nn.sigmoid(self.fc2(self.fc1(avg)) + self.fc2(self.fc1(mx)))
        x = x * cw
        a = tf.reduce_mean(x, axis=-1, keepdims=True)
        m = tf.reduce_max(x, axis=-1, keepdims=True)
        return x * self.sp(tf.concat([a, m], axis=-1))
class CoordinateAttention(layers.Layer):
    def __init__(self, reduction=32, **kwargs):
        super().__init__(**kwargs); self.reduction = reduction
    def build(self, input_shape):
        c = int(input_shape[-1]); mid = max(c // self.reduction, 8)
        self.c1 = layers.Conv2D(mid, 1, padding='same', use_bias=False)
        self.bn = layers.BatchNormalization()
        self.ch = layers.Conv2D(c, 1, padding='same', activation='sigmoid')
        self.cw = layers.Conv2D(c, 1, padding='same', activation='sigmoid')
        super().build(input_shape)
    def call(self, x):
        zh = tf.reduce_mean(x, axis=2, keepdims=True)
        zw = tf.transpose(tf.reduce_mean(x, axis=1, keepdims=True), [0, 2, 1, 3])
        z = tf.nn.relu(self.bn(self.c1(tf.concat([zh, zw], axis=1))))
        h1 = tf.shape(zh)[1]
        fh, fw = z[:, :h1], z[:, h1:]
        fw = tf.transpose(fw, [0, 2, 1, 3])
        return x * self.ch(fh) * self.cw(fw)
class VariableSpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.conv = layers.Conv2D(1, kernel_size, padding='same', activation='sigmoid')
    def call(self, x):
        a = tf.reduce_mean(x, axis=-1, keepdims=True)
        m = tf.reduce_max(x, axis=-1, keepdims=True)
        return x * self.conv(tf.concat([a, m], axis=-1))
print("✓ Extended attention modules loaded (SE/ECA/CBAM/Coordinate/VariableSpatial)")

## SECTION 1: MODEL CONSTRUCTION UTILITIES

In [ ]:
# Cell 3: Model construction with TRUE stage-wise attention insertion
INSERTION_LAYERS = {
    'P1': 'conv2_block3_out',   # 56x56,   256 ch (high-res early stage)
    'P2': 'conv3_block4_out',   # 28x28,   512 ch
    'P3': 'conv4_block6_out',   # 14x14,  1024 ch
    'P4': 'conv5_block3_out',   # 7x7,    2048 ch
}

def apply_attention_by_type(x, attention_type, spatial_kernel=7, reduction_ratio=16):
    """Apply one attention block. 'none' = identity (baseline A0)."""
    if attention_type == 'none':
        return x
    if attention_type == 'channel':
        return ChannelAttention(reduction_ratio=reduction_ratio)(x)
    if attention_type == 'spatial':
        return SpatialAttention(kernel_size=spatial_kernel)(x)
    if attention_type == 'channel_spatial':
        x = ChannelAttention(reduction_ratio=reduction_ratio)(x)
        return SpatialAttention(kernel_size=spatial_kernel)(x)
    if attention_type == 'spatial_channel':
        x = SpatialAttention(kernel_size=spatial_kernel)(x)
        return ChannelAttention(reduction_ratio=reduction_ratio)(x)
    if attention_type == 'self':
        return SelfAttention()(x)
    if attention_type == 'combined':
        x = ChannelAttention(reduction_ratio=reduction_ratio)(x)
        x = SpatialAttention(kernel_size=spatial_kernel)(x)
        return SelfAttention()(x)
    if attention_type == 'se':
        return SEAttention(reduction_ratio=reduction_ratio)(x)
    if attention_type == 'eca':
        return ECAAttention()(x)
    if attention_type == 'cbam':
        return CBAM()(x)
    if attention_type == 'coordinate':
        return CoordinateAttention()(x)
    if attention_type == 'variable_spatial':
        return VariableSpatialAttention(kernel_size=spatial_kernel)(x)
    raise ValueError(f"Unknown attention_type: {attention_type}")

def build_model_with_attention(attention_type='channel', position='P3', spatial_kernel=7,
                               reduction_ratio=16, input_shape=(224, 224, 3), freeze_base=True):
    """
    Build ResNet50 with attention inserted INSIDE the backbone at stage `position`.
    The backbone graph is replayed layer-by-layer: layers before the insertion point
    run as usual, the attention block transforms the intermediate feature map, and
    the REMAINING backbone stages consume the attended features (true ablation,
    not attention-after-backbone).
    """
    if position not in INSERTION_LAYERS:
        raise ValueError(f"Position must be one of {list(INSERTION_LAYERS)}")
    insertion_layer = INSERTION_LAYERS[position]
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = not freeze_base if False else False  # initially frozen; unfreeze later if needed
    names = [l.name for l in base.layers]
    if insertion_layer not in names:
        raise ValueError(f"Layer {insertion_layer} not found in ResNet50")
    cut = names.index(insertion_layer)
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for layer in base.layers[:cut + 1]:
        if isinstance(layer, layers.InputLayer):
            continue
        x = layer(x)
    # --- TRUE insertion: attend the intermediate feature map ---
    x = apply_attention_by_type(x, attention_type, spatial_kernel, reduction_ratio)
    # --- Remaining backbone stages consume ATTENDED features ---
    for layer in base.layers[cut + 1:]:
        x = layer(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss=CategoricalCrossentropy(), metrics=['accuracy'])
    return model

def build_model_multi_position(attention_type='channel_spatial', positions=('P2', 'P3', 'P4'),
                               spatial_kernel=7, reduction_ratio=16,
                               input_shape=(224, 224, 3)):
    """Insert attention at MULTIPLE stages (Experiment 3, model C4)."""
    for p in positions:
        if p not in INSERTION_LAYERS:
            raise ValueError(f"Position must be one of {list(INSERTION_LAYERS)}")
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False
    names = [l.name for l in base.layers]
    cuts = sorted(names.index(INSERTION_LAYERS[p]) for p in positions)
    cut_set = set(cuts)
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for i, layer in enumerate(base.layers):
        if isinstance(layer, layers.InputLayer):
            continue
        x = layer(x)
        if i in cut_set:
            x = apply_attention_by_type(x, attention_type, spatial_kernel, reduction_ratio)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss=CategoricalCrossentropy(), metrics=['accuracy'])
    return model

print("✓ Model construction functions defined (TRUE stage-wise insertion)")
print(f"  Insertion stages: {INSERTION_LAYERS}")

In [ ]:
# Cell 4: Test model construction
test_model = build_model_with_attention(attention_type='channel', position='P3')
print("\nTest Model Summary:")
test_model.summary()
print(f"\n✓ Model built successfully")

## SECTION 2: EXPERIMENT 1 - ATTENTION TYPE ABLATION (FIXED AT P3)

In [ ]:
# Cell 5: Define training configuration (IDENTICAL FOR ALL MODELS)
BATCH_SIZE = 32
EPOCHS = 30
INITIAL_LR = 1e-3
FINETUNE_LR = 1e-4
VALIDATION_SPLIT = 0.2

training_config = {
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'initial_lr': INITIAL_LR,
    'finetune_lr': FINETUNE_LR,
    'validation_split': VALIDATION_SPLIT,
    'optimizer': 'Adam',
    'loss': 'CategoricalCrossentropy'
}

print("Training Configuration (IDENTICAL FOR ALL MODELS):")
print("="*50)
for key, val in training_config.items():
    print(f"  {key}: {val}")

# Save config
with open('results/training_config.pkl', 'wb') as f:
    pickle.dump(training_config, f)

In [ ]:
# Cell 6: Define Experiment 1 models (Attention Type Ablation at P3)
exp1_models = {
    'A0_Baseline': {'type': 'none', 'position': 'P3'},
    'A1_Channel': {'type': 'channel', 'position': 'P3'},
    'A2_Spatial': {'type': 'spatial', 'position': 'P3'},
    'A3_ChannelSpatial': {'type': 'channel_spatial', 'position': 'P3'},
    'A4_Self': {'type': 'self', 'position': 'P3'},
    'A5_Combined': {'type': 'combined', 'position': 'P3'}
}

print("Experiment 1: Attention-Type Ablation (Fixed Position P3)")
print("="*60)
for model_name, config in exp1_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(exp1_models)}")

In [ ]:
# Cell 7: Training function with early stopping and learning rate scheduling
def train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=30):
    """
    Train model with validation and save results
    """
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=0
        )
    ]
    
    # Train
    print(f"\nTraining {model_name}...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )
    
    # Save model and history
    model.save(f'results/models/{model_name}_model.h5')
    with open(f'results/models/{model_name}_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)
    
    print(f"  ✓ Training completed (Epochs: {len(history.history['loss'])})")
    print(f"    Final train acc: {history.history['accuracy'][-1]:.4f}")
    print(f"    Final val acc: {history.history['val_accuracy'][-1]:.4f}")
    
    return history

print("✓ Training function defined")

In [ ]:
# Cell 8: Train all Experiment 1 models (MAIN TRAINING LOOP)
exp1_histories = {}
exp1_trained_models = {}

for idx, (model_name, config) in enumerate(exp1_models.items(), 1):
    print(f"\n[{idx}/{len(exp1_models)}] Building {model_name}...")
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp1_histories[model_name] = history.history
    exp1_trained_models[model_name] = model

print("\n" + "="*60)
print("✓ EXPERIMENT 1 COMPLETE: All models trained")
print("="*60)

## SECTION 3: EXPERIMENT 2 - ATTENTION POSITION ABLATION

In [ ]:
# Cell 9: Define Experiment 2 models (Attention Position Ablation)
exp2_models = {
    'B2_P2': {'type': 'channel_spatial', 'position': 'P2'},
    'B3_P3': {'type': 'channel_spatial', 'position': 'P3'},
    'B4_P4': {'type': 'channel_spatial', 'position': 'P4'}
}

print("Experiment 2: Attention-Position Ablation (Fixed Type: Channel-Spatial)")
print("="*70)
for model_name, config in exp2_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(exp2_models)}")
print("Note: B1 (P1) skipped due to high spatial resolution (computational cost)")

In [ ]:
# Cell 10: Train all Experiment 2 models
exp2_histories = {}
exp2_trained_models = {}

for idx, (model_name, config) in enumerate(exp2_models.items(), 1):
    print(f"\n[{idx}/{len(exp2_models)}] Building {model_name}...")
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp2_histories[model_name] = history.history
    exp2_trained_models[model_name] = model

print("\n" + "="*60)
print("✓ EXPERIMENT 2 COMPLETE: All position models trained")
print("="*60)

In [ ]:
# Cell 11: Save all experiment histories and models
# Save Experiment 1
with open('results/exp1_histories.pkl', 'wb') as f:
    pickle.dump(exp1_histories, f)

# Save Experiment 2
with open('results/exp2_histories.pkl', 'wb') as f:
    pickle.dump(exp2_histories, f)

print("✓ Experiment histories saved")
print(f"  Experiment 1: {len(exp1_histories)} models")
print(f"  Experiment 2: {len(exp2_histories)} models")

## SECTION 4: EXPERIMENT 3 - MULTI-POSITION ATTENTION (TYPE FIXED: CHANNEL-SPATIAL)
Single-position B2/B3/B4 (Exp. 2) are reused; here attention is inserted at MULTIPLE stages at once.

In [ ]:
# Cell 12: Define Experiment 3 models (multi-position insertion)
exp3_models = {
    'C1_P2P3': {'type': 'channel_spatial', 'positions': ('P2', 'P3')},
    'C2_P3P4': {'type': 'channel_spatial', 'positions': ('P3', 'P4')},
    'C3_P2P3P4': {'type': 'channel_spatial', 'positions': ('P2', 'P3', 'P4')},
}
print("Experiment 3: Multi-Position Attention (Fixed Type: Channel-Spatial)")
print("=" * 70)
for n, c in exp3_models.items():
    print(f"  {n}: Type={c['type']}, Positions={c['positions']}")

In [ ]:
# Cell 13: Train all Experiment 3 models
exp3_histories = {}; exp3_trained_models = {}
for idx, (model_name, config) in enumerate(exp3_models.items(), 1):
    print(f"\n[{idx}/{len(exp3_models)}] Building {model_name}...")
    model = build_model_multi_position(attention_type=config['type'],
                                       positions=config['positions'])
    history = train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=EPOCHS)
    exp3_histories[model_name] = history.history
    exp3_trained_models[model_name] = model
with open('results/exp3_histories.pkl', 'wb') as f:
    pickle.dump(exp3_histories, f)
print("\n" + "=" * 60)
print("✓ EXPERIMENT 3 COMPLETE")
print("=" * 60)

## SECTION 5: EXPERIMENT 4 - ATTENTION ORDER ABLATION (FIXED AT P3)
D1 = Channel->Spatial vs D2 = Spatial->Channel. Tests whether order matters.

In [ ]:
# Cell 14: Define + train Experiment 4 models (order ablation)
exp4_models = {
    'D1_ChannelSpatial': {'type': 'channel_spatial', 'position': 'P3'},
    'D2_SpatialChannel': {'type': 'spatial_channel', 'position': 'P3'},
}
exp4_histories = {}; exp4_trained_models = {}
for idx, (model_name, config) in enumerate(exp4_models.items(), 1):
    print(f"\n[{idx}/{len(exp4_models)}] Building {model_name}...")
    model = build_model_with_attention(attention_type=config['type'],
                                       position=config['position'])
    history = train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=EPOCHS)
    exp4_histories[model_name] = history.history
    exp4_trained_models[model_name] = model
with open('results/exp4_histories.pkl', 'wb') as f:
    pickle.dump(exp4_histories, f)
print("\n" + "=" * 60)
print("✓ EXPERIMENT 4 COMPLETE")
print("=" * 60)

## SECTION 6: EXPERIMENT 5 - SPATIAL KERNEL SIZE ABLATION (FIXED AT P3)
E1 (3x3) vs E2 (5x5) vs E3 (7x7) using VariableSpatialAttention.

In [ ]:
# Cell 15: Define + train Experiment 5 models (kernel ablation)
exp5_models = {
    'E1_Kernel3': {'type': 'variable_spatial', 'position': 'P3', 'kernel': 3},
    'E2_Kernel5': {'type': 'variable_spatial', 'position': 'P3', 'kernel': 5},
    'E3_Kernel7': {'type': 'variable_spatial', 'position': 'P3', 'kernel': 7},
}
exp5_histories = {}; exp5_trained_models = {}
for idx, (model_name, config) in enumerate(exp5_models.items(), 1):
    print(f"\n[{idx}/{len(exp5_models)}] Building {model_name} (kernel={config['kernel']})...")
    model = build_model_with_attention(attention_type=config['type'],
                                       position=config['position'],
                                       spatial_kernel=config['kernel'])
    history = train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=EPOCHS)
    exp5_histories[model_name] = history.history
    exp5_trained_models[model_name] = model
with open('results/exp5_histories.pkl', 'wb') as f:
    pickle.dump(exp5_histories, f)
print("\n" + "=" * 60)
print("✓ EXPERIMENT 5 COMPLETE")
print("=" * 60)

## SECTION 7: EXTENDED EXPERIMENT - ADDITIONAL ATTENTION MECHANISMS (FIXED AT P3)
F1 = SE, F2 = ECA, F3 = CBAM, F4 = Coordinate Attention. Compares published mechanisms.

In [ ]:
# Cell 16: Define + train additional-mechanism models
additional_models = {
    'F1_SE': {'type': 'se', 'position': 'P3'},
    'F2_ECA': {'type': 'eca', 'position': 'P3'},
    'F3_CBAM': {'type': 'cbam', 'position': 'P3'},
    'F4_Coordinate': {'type': 'coordinate', 'position': 'P3'},
}
additional_histories = {}; additional_trained_models = {}
for idx, (model_name, config) in enumerate(additional_models.items(), 1):
    print(f"\n[{idx}/{len(additional_models)}] Building {model_name}...")
    model = build_model_with_attention(attention_type=config['type'],
                                       position=config['position'])
    history = train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=EPOCHS)
    additional_histories[model_name] = history.history
    additional_trained_models[model_name] = model
with open('results/additional_histories.pkl', 'wb') as f:
    pickle.dump(additional_histories, f)
print("\n" + "=" * 60)
print("✓ EXTENDED EXPERIMENT COMPLETE")
print("=" * 60)

---

**Checklist Progress**:
- [x] Models constructed with TRUE stage-wise attention insertion (P1/P2/P3/P4)
- [x] Training configuration defined (IDENTICAL FOR ALL)
- [x] Experiment 1: Attention-Type Ablation (6 models at P3)
- [x] Experiment 2: Attention-Position Ablation (3 models, Channel-Spatial)
- [x] Experiment 3: Multi-Position Attention (3 models)
- [x] Experiment 4: Attention Order Ablation (2 models)
- [x] Experiment 5: Spatial Kernel Ablation (3 models)
- [x] Extended: Additional Mechanisms SE/ECA/CBAM/Coordinate (4 models)
- [ ] Evaluation metrics and result tables
- [ ] Visualizations (plots and graphs)
- [ ] Grad-CAM analysis

**Next: Part 4 - Evaluation Metrics and Result Tables**